# 03.04 · Word del insumo de Evidencia + Recomendaciones

Arma un Word por IDO PPI con las tres secciones del insumo, sin llamar a Azure. El formato
es el de los insumos hechos a mano de CAF: se parte de la plantilla
`references/plantilla_evid_recom.docx` (encabezado con logos, márgenes y estilos).

| Parte del Word | Origen |
|---|---|
| Título, sector, fecha y resumen de la operación | `03.02` (la fecha es la de generación) |
| **Sección 1.** Evidencia para mejorar la estructuración del proyecto | `03.02` |
| **Sección 2.** Implicaciones para la operación, con semáforo por viñeta | `03.03` |
| **Sección 3.** Recomendación para explorar posible evaluación (Sí/No y concepto) | Paso 2: decisión y concepto finales del `02.00` |
| Referencias | `03.02` |

La entrada es `idos_seccion2.csv` del `03.03`, que ya trae todas las columnas anteriores. Los
Word se guardan en `data/processed/3. Insumo Evid+Recom/` y **se reescriben en cada ejecución**:
lo que se edite a mano debe hacerse en una copia.

## 1. LOAD LIBRARIES

`python-docx` escribe el Word. Las rutas se toman de `caf_input_cci_preparatoria/config.py`.

In [ ]:
from datetime import date
import json
from pathlib import Path
import re
import sys

from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.opc.constants import RELATIONSHIP_TYPE
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Cm, Pt, RGBColor
from IPython.display import display
from lxml import etree
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))  # para importar el paquete caf_input_cci_preparatoria

from caf_input_cci_preparatoria import config

## 2. LOAD DATA

Se lee la tabla del `03.03` como texto y se convierten a listas las columnas guardadas en JSON.
La plantilla del Word está en `references/`.

In [ ]:
archivo_entrada = (
    config.INTERIM / "3. Insumo Evid+Recom" / "03.03_seccion2_implicaciones" / "idos_seccion2.csv"
)
carpeta_salida = config.PROCESSED / "3. Insumo Evid+Recom"
archivo_plantilla = config.REFERENCES / "plantilla_evid_recom.docx"
assert archivo_plantilla.is_file(), f"Falta la plantilla {archivo_plantilla}"

idos = pd.read_csv(archivo_entrada, encoding="utf-8-sig", dtype=str, keep_default_na=False)
idos.index = pd.RangeIndex(1, len(idos) + 1)
for campo in ["impactos", "consideraciones_diseno", "referencias", "bloques"]:
    idos[campo] = idos[campo].map(json.loads)
print(f"{len(idos)} IDOs con las secciones 1 y 2")

## 3. PROCESS DATA

### Formato

Cada Word parte de `plantilla_evid_recom.docx`, un insumo hecho a mano (Caranavi–Guanay): se
borra su contenido y se conservan el encabezado (logos de Evaluación de Impacto y CAF y la
leyenda «DADMI · Brief de evidencia y recomendaciones · Uso interno CAF»), los márgenes y los
estilos. La plantilla no se modifica. Para cambiar el encabezado o los logos, se reemplaza ese
archivo.

| Elemento | Formato |
|---|---|
| Texto y viñetas | Aptos 11, idioma español de Colombia (`es-CO`) para la ortografía |
| Título de la operación | Negrita, azul CAF (`0A4A7A`), 14 pt |
| Títulos de sección (evidencia, implicaciones, recomendación, referencias) | Negrita blanca sobre barra azul CAF, 14 pt |
| Subtítulo de sector | Negrita, verde azulado (`31849B`), 14 pt |
| Fecha | Gris, 10 pt, sin cursiva |
| Nivel de confianza y cobertura | Nota al pie del título de la sección 1, con los rótulos en negrita |
| Subtítulos (impactos, condiciones y bloques de la sección 2) | Texto en negrita verde azulado (`1A998B`), 12 pt, sin fondo |
| Sección 3 | Título de sección, subtítulo verde «Decisión: Sí/No» y el concepto en texto normal |

Espacios: 12 pt antes de cada título de sección, 8 pt antes de cada subtítulo, 4 pt después de
cada párrafo y viñeta.

In [3]:
AZUL = RGBColor(0x0A, 0x4A, 0x7A)
VERDE_AZULADO = RGBColor(0x31, 0x84, 0x9B)
GRIS = RGBColor(0x80, 0x80, 0x80)
VERDE_SUBTITULO = RGBColor(0x1A, 0x99, 0x8B)
BLANCO = RGBColor(0xFF, 0xFF, 0xFF)
FONDO_SECCION = "0A4A7A"  # azul CAF
SEMAFORO = {  # color: (símbolo, leyenda)
    "Rojo": ("🔴", "Atención prioritaria"),
    "Amarillo": ("🟡", "Completar o precisar"),
    "Verde": ("🟢", "En orden, verificar"),
}
NIVELES = ["Alta", "Media-alta", "Media-baja", "Baja"]
MESES = [
    "Enero",
    "Febrero",
    "Marzo",
    "Abril",
    "Mayo",
    "Junio",
    "Julio",
    "Agosto",
    "Septiembre",
    "Octubre",
    "Noviembre",
    "Diciembre",
]
fecha_texto = f"{MESES[date.today().month - 1]} {date.today().day}, {date.today().year}"


def nuevo_documento():
    """Abre la plantilla y borra su contenido; quedan encabezado, márgenes y estilos."""
    documento = Document(archivo_plantilla)
    cuerpo = documento.element.body
    for elemento in list(cuerpo):
        if elemento.tag != qn("w:sectPr"):
            cuerpo.remove(elemento)
    normal = documento.styles["Normal"]
    normal.font.name, normal.font.size = "Aptos", Pt(11)
    normal.paragraph_format.space_after = Pt(4)
    for idioma in documento.styles.element.iter(qn("w:lang")):  # la plantilla viene en inglés
        idioma.set(qn("w:val"), "es-CO")
    return documento

### Bloques del Word

Funciones cortas para cada tipo de párrafo: títulos, subtítulos en verde, párrafo justificado,
viñeta con el tema en negrita y notas al pie. El nivel de confianza y la cobertura van en una
nota al pie del título de la sección 1. En la sección 2, el
símbolo del semáforo (🔴 🟡 🟢) reemplaza la viñeta y su leyenda va en una nota al pie del
título «Implicaciones para la operación» (python-docx no crea notas al pie, así que se agrega
directo en el XML de la plantilla). Solo las rojas llevan, a continuación del texto
y con el rótulo en negrita, «Qué falta:» con la justificación del semáforo; así se destacan sin
alargar el documento.

In [4]:
def titulo(documento, texto, tamano=14, espacio_antes=12, color=AZUL):
    parrafo = documento.add_paragraph()
    parrafo.paragraph_format.space_before = Pt(espacio_antes)
    parrafo.paragraph_format.space_after = Pt(4)
    parrafo.paragraph_format.keep_with_next = True
    run = parrafo.add_run(texto)
    run.bold, run.font.size, run.font.color.rgb = True, Pt(tamano), color
    return parrafo


def seccion(documento, texto):
    """Título de sección: texto blanco sobre barra azul CAF."""
    nuevo = titulo(documento, texto, color=BLANCO)
    sombra = OxmlElement("w:shd")
    sombra.set(qn("w:val"), "clear")
    sombra.set(qn("w:fill"), FONDO_SECCION)
    nuevo._p.get_or_add_pPr().append(sombra)
    return nuevo


def subtitulo(documento, texto):
    return titulo(documento, texto, tamano=12, espacio_antes=8, color=VERDE_SUBTITULO)


def parrafo(documento, texto):
    nuevo = documento.add_paragraph(texto)
    nuevo.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    return nuevo


def vineta(documento, tema, texto):
    nuevo = documento.add_paragraph(style="List Bullet")
    nuevo.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    nuevo.paragraph_format.left_indent = Cm(0.63)  # sangría francesa, como la plantilla
    nuevo.paragraph_format.first_line_indent = Cm(-0.63)
    if tema:
        nuevo.add_run(f"{tema.rstrip(':. ')}: ").bold = True
    nuevo.add_run(texto)
    return nuevo


def vineta_semaforo(documento, item):
    nuevo = documento.add_paragraph()
    nuevo.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    nuevo.paragraph_format.left_indent = Cm(0.63)
    nuevo.paragraph_format.first_line_indent = Cm(-0.63)
    nuevo.add_run(f"{SEMAFORO[item['semaforo']][0]}\t")  # tabulación: no se estira al justificar
    if item["tema"]:
        nuevo.add_run(f"{item['tema'].rstrip(':. ')}: ").bold = True
    nuevo.add_run(item["texto"])
    if item["semaforo"] == "Rojo":
        nuevo.add_run(" Qué falta: ").bold = True
        nuevo.add_run(item["justificacion_semaforo"])
    return nuevo


def nota_al_pie(documento, parrafo, tramos):
    """Agrega una nota al pie y pone su número al final de `parrafo`.

    `tramos` es una lista de pares (texto, negrita).
    """
    parte = documento.part.part_related_by(RELATIONSHIP_TYPE.FOOTNOTES)
    notas = etree.fromstring(parte.blob)
    numero = str(max(int(nota.get(qn("w:id"))) for nota in notas) + 1)

    def elemento(padre, etiqueta, **atributos):
        nuevo = etree.SubElement(padre, qn(f"w:{etiqueta}"))
        for clave, valor in atributos.items():
            nuevo.set(qn(f"w:{clave}"), valor)
        return nuevo

    nota = elemento(notas, "footnote", id=numero)
    linea = elemento(nota, "p")
    elemento(elemento(linea, "pPr"), "pStyle", val="FootnoteText")
    marca = elemento(linea, "r")
    elemento(elemento(marca, "rPr"), "rStyle", val="FootnoteReference")
    elemento(marca, "footnoteRef")
    for texto, negrita in [(" ", False), *tramos]:
        contenido = elemento(linea, "r")
        formato = elemento(contenido, "rPr")
        elemento(formato, "rFonts", ascii="Aptos", hAnsi="Aptos")
        if negrita:
            elemento(formato, "b")
        elemento(formato, "sz", val="18")
        escrito = elemento(contenido, "t")
        escrito.text = texto
        escrito.set("{http://www.w3.org/XML/1998/namespace}space", "preserve")
    parte._blob = etree.tostring(notas, xml_declaration=True, encoding="UTF-8", standalone=True)

    base = parrafo.runs[0]  # el número toma el color, el tamaño y la negrita del título
    referencia = parrafo.add_run()
    elemento(elemento(referencia._r, "rPr"), "rStyle", val="FootnoteReference")
    referencia.bold, referencia.font.size = base.bold, base.font.size
    referencia.font.color.rgb = base.font.color.rgb
    elemento(referencia._r, "footnoteReference", id=numero)


def tramos_confianza(ido):
    """Nivel de confianza y cobertura geográfica, para la nota al pie de la sección 1."""
    tramos = [("Nivel de confianza de la evidencia:  ", True)]
    tramos += [(casillas(NIVELES, ido["nivel_confianza"]) + ".  ", False)]
    tramos += [("Cobertura geográfica de la evidencia: ", True)]
    tramos += [(ido["cobertura_geografica"], False)]
    nota = nota_confianza(ido["detalle_confianza"])
    if nota:
        tramos += [(f". Confianza por tema: {nota}", False)]
    return tramos


def casillas(opciones, marcada):
    return "    ".join(f"{'☒' if opcion == marcada else '☐'} {opcion}" for opcion in opciones)

### Nivel de confianza

Se marca la casilla calculada en el `03.02`. Si las filas del repositorio tienen niveles
distintos, se agrega entre paréntesis el nivel de cada tema, sin números de fila.

In [5]:
def nota_confianza(detalle):
    niveles = re.findall(r"Fila \d+ \((.+?)\): ([\w-]+(?: \w+)?)", detalle)
    if len({nivel for _, nivel in niveles}) <= 1:
        return ""
    return "(" + "; ".join(f"{tema}: {nivel}" for tema, nivel in niveles) + ")"

### Decisión y concepto de la sección 3

Se usan la decisión y el concepto finales que se aprobaron o editaron en el `02.00`
(`decision_final` y `concepto_final`). Si un IDO quedó **Pendiente** (sin aprobar), se usa el
borrador de Azure (`decision_sugerida` y `concepto`) y el Word lo marca como borrador.

In [ ]:
pendiente = idos["estado_concepto"].eq("Pendiente")
idos["decision_seccion3"] = idos["decision_final"].mask(pendiente, idos["decision_sugerida"])
idos["concepto_seccion3"] = idos["concepto_final"].mask(pendiente, idos["concepto"])
print(f"{pendiente.sum()} IDOs con el concepto sin aprobar: su Word lleva el borrador de Azure")

### Armar el Word de un IDO

El orden reproduce el insumo manual. La sección 3 usa `decision_seccion3` y
`concepto_seccion3`; si el concepto está pendiente, agrega la marca «Borrador: concepto sin
aprobar» antes del texto.

In [6]:
def armar_word(ido):
    documento = nuevo_documento()
    titulo(documento, ido["operacion"], espacio_antes=0)
    titulo(documento, ido["subtitulo_sector"], espacio_antes=0, color=VERDE_AZULADO)
    fecha = documento.add_paragraph().add_run(fecha_texto)
    fecha.font.size, fecha.font.color.rgb = Pt(10), GRIS
    parrafo(documento, ido["resumen_operacion"]).paragraph_format.space_before = Pt(6)

    # Sección 1
    evidencia = seccion(documento, "Evidencia para mejorar la estructuración del proyecto")
    nota_al_pie(documento, evidencia, tramos_confianza(ido))
    if ido["impactos"]:
        subtitulo(documento, ido["titulo_impactos"])
        parrafo(documento, ido["introduccion_impactos"])
        for impacto in ido["impactos"]:
            vineta(documento, impacto["tema"], impacto["texto"])
    subtitulo(documento, ido["titulo_consideraciones"])
    for consideracion in ido["consideraciones_diseno"]:
        vineta(documento, consideracion["tema"], consideracion["texto"])

    # Sección 2
    implicaciones = seccion(documento, "Implicaciones para la operación")
    leyenda = "     ".join(f"{simbolo} {texto}" for simbolo, texto in SEMAFORO.values())
    nota_al_pie(documento, implicaciones, [(leyenda, False)])
    for bloque in ido["bloques"]:
        subtitulo(documento, bloque["titulo"])
        for item in bloque["vinetas"]:
            vineta_semaforo(documento, item)

    # Sección 3
    seccion(documento, "Recomendación para explorar posible evaluación de la operación")
    subtitulo(documento, f"Decisión: {ido['decision_seccion3'] or 'sin definir'}")
    if ido["estado_concepto"] == "Pendiente":
        documento.add_paragraph().add_run("Borrador: concepto sin aprobar").bold = True
    parrafo(documento, ido["concepto_seccion3"])

    # Referencias
    if ido["referencias"]:
        seccion(documento, "Referencias")
        for referencia in ido["referencias"]:
            run = documento.add_paragraph().add_run(referencia)
            run.font.size = Pt(9)
    return documento

### Nombre de cada Word

`MM DD AA - Evid+Recom - <operación>, <país>.docx` (por ejemplo, `11 04 26 - …`), como los insumos hechos a mano. La fecha
es la de registro del IDO en el paso 2 (`fecha_registro`); si falta, dice «sin fecha». El país
se acorta quitando el nombre oficial («República del Perú» → «Perú», «Estado Plurinacional de
Bolivia» → «Bolivia»). La operación va sin comillas ni caracteres que no admiten los nombres de
archivo, recortada en un espacio para no pasar de 90 caracteres. Una nueva ejecución reemplaza
el Word del mismo IDO mientras no cambie su fecha de registro.

In [7]:
def limpiar_nombre(texto):
    limpio = re.sub(r'[“”"\'«»/\\:*?<>|]', "", texto)
    return re.sub(r"\s+", " ", limpio).replace(" ,", ",").strip(" ,")


def pais_corto(pais):
    oficial = r"^(Estado Plurinacional de|República (Bolivariana |Oriental )?del?)\s+"
    return re.sub(oficial, "", limpiar_nombre(pais), flags=re.IGNORECASE)


def nombre_archivo(ido):
    operacion = limpiar_nombre(ido["operacion"])
    if len(operacion) > 90:  # se corta en un espacio y sin palabras sueltas al final
        operacion = operacion[:90].rsplit(" ", 1)[0]
        operacion = re.sub(r"(\s+(de|del|la|las|el|los|y|e|en|para|a|con))+$", "", operacion)
        operacion = operacion.rstrip(" -–,.;")
    fecha = pd.to_datetime(ido["fecha_registro"], errors="coerce")
    fecha = "sin fecha" if pd.isna(fecha) else fecha.strftime("%m %d %y")
    pais = pais_corto(ido["pais"])
    return f"{fecha} - Evid+Recom - {operacion}{', ' + pais if pais else ''}.docx"


idos["word"] = [nombre_archivo(ido) for ido in idos.to_dict("records")]
repetidos = idos.loc[idos["word"].duplicated(keep=False), ["archivo", "word"]]
if not repetidos.empty:
    print("Aviso: IDOs con el mismo nombre de Word; se agrega el número de IDO:")
    display(repetidos)
    idos.loc[repetidos.index, "word"] = [
        nombre.replace(".docx", f" ({numero}).docx")
        for numero, nombre in repetidos["word"].items()
    ]

Aviso: IDOs con el mismo nombre de Word; se agrega el número de IDO:


,archivo,word
7,IDO Cierre de Brechas Chancay V18Ago2026 copy....,Evid+Recom - Proyecto de Mejoramiento y amplia...
8,IDO Cierre de Brechas Chancay V18Ago2026.docx,Evid+Recom - Proyecto de Mejoramiento y amplia...


### Control antes de guardar

Se muestra, por IDO, la casilla de confianza, la decisión de la sección 3 y su estado, los rojos del semáforo, la extensión
aproximada del texto y los avisos de los notebooks anteriores que conviene revisar en el Word:
citas no localizadas, IDO sin impactos (fila del repositorio sin contenido de evidencia) y
vacíos del IDO que la misión debería completar. Unas 550 palabras equivalen a una página.

In [8]:
def palabras(ido):
    textos = [ido["resumen_operacion"], ido["introduccion_impactos"], ido["concepto_seccion3"]]
    textos += [v["texto"] for v in ido["impactos"] + ido["consideraciones_diseno"]]
    textos += [v["texto"] for bloque in ido["bloques"] for v in bloque["vinetas"]]
    return sum(len(texto.split()) for texto in textos)


control = pd.DataFrame(
    {
        "word": idos["word"],
        "confianza": idos["nivel_confianza"],
        "decision": idos["decision_seccion3"],
        "estado_concepto": idos["estado_concepto"],
        "palabras": [palabras(ido) for ido in idos.to_dict("records")],
        "rojos": idos["rojos"],
        "sin_impactos": idos["impactos"].map(lambda lista: "Sí" if not lista else ""),
        "citas_por_revisar": idos["citas_no_localizadas"] + idos["citas_no_localizadas_seccion2"],
    }
)
with pd.option_context("display.max_colwidth", 70):
    display(control)

,word,confianza,decision,palabras,sin_impactos,citas_por_revisar
1,Evid+Recom - Proyecto de Construcción de la Línea de Transmisión 5...,Alta,No,1124,,
2,Evid+Recom - Componente de Riego Proyecto Múltiple San Jacinto (Mo...,Alta,No,1224,,
3,Evid+Recom - Programa de Impulso a la Oferta de Vivienda Social.docx,Media-alta,No,1233,,
4,Evid+Recom - Programa OSE 4.0.docx,Media-baja,No,1018,Sí,
5,Evid+Recom - Programa El Salvador Vuela – Fase II.docx,Media-baja,No,1184,,
6,Evid+Recom - Proyecto Construcción Carretera Caranavi – Guanay Tra...,Alta,No,1157,,
7,Evid+Recom - Proyecto de Mejoramiento y ampliación del servicio de...,Alta,No,1177,,
8,Evid+Recom - Proyecto de Mejoramiento y ampliación del servicio de...,Alta,No,1153,,
9,Evid+Recom - Programa de Infraestructura Vial y Conectividad Turís...,Alta,No,1159,,
10,Evid+Recom - Repotenciación del lado aire del aeropuerto San Crist...,Media-baja,No,1120,,


In [9]:
ido_a_revisar = 1
print(f"Vacíos del IDO {ido_a_revisar} que la misión debería completar:")
print(idos.at[ido_a_revisar, "vacios_ido"] or "ninguno")

Vacíos del IDO 1 que la misión debería completar:
En la sección 4.1, el apartado de riesgo soberano contiene únicamente instrucciones y no presenta resultados de SECAPS; la misión debe completar la calificación global y por Área Crítica de Control.


## 4. SAVE DATA

Se guarda un Word por IDO en `data/processed/3. Insumo Evid+Recom/` (la carpeta se crea si no
existe). Los Word de IDOs que ya no están en la tabla no se borran.

In [10]:
carpeta_salida.mkdir(parents=True, exist_ok=True)
for ido in idos.to_dict("records"):
    armar_word(ido).save(carpeta_salida / ido["word"])
print(f"{len(idos)} Word guardados en {carpeta_salida.relative_to(ROOT)}:")
print("\n".join(idos["word"]))

12 Word guardados en data/processed/3. Insumo Evid+Recom:
Evid+Recom - Proyecto de Construcción de la Línea de Transmisión 500 kV Emboscada – Horqueta y de la.docx
Evid+Recom - Componente de Riego Proyecto Múltiple San Jacinto (Modernización – Tecnificación).docx
Evid+Recom - Programa de Impulso a la Oferta de Vivienda Social.docx
Evid+Recom - Programa OSE 4.0.docx
Evid+Recom - Programa El Salvador Vuela – Fase II.docx
Evid+Recom - Proyecto Construcción Carretera Caranavi – Guanay Tramo I.docx
Evid+Recom - Proyecto de Mejoramiento y ampliación del servicio de agua potable urbano y mejoramiento (7).docx
Evid+Recom - Proyecto de Mejoramiento y ampliación del servicio de agua potable urbano y mejoramiento (8).docx
Evid+Recom - Programa de Infraestructura Vial y Conectividad Turística en el Destino Turístico Cusco.docx
Evid+Recom - Repotenciación del lado aire del aeropuerto San Cristóbal de Galápagos (RASCGA).docx
Evid+Recom - Programa Nacional de Obras Viales.docx
Evid+Recom - Clima 360°